# Statistical Comparison of Multilayer Networks

This notebook demonstrates how to use py3plex's statistical comparison framework to compare multilayer networks.

## Overview

The `compare_multilayer_networks` function enables:
- Pairwise and multi-group comparisons
- Multiple statistical tests (parametric, non-parametric, permutation)
- Effect size estimation
- Multiple comparison correction
- Bootstrap confidence intervals

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from py3plex.core import multinet
from py3plex.datasets import make_random_multiplex
from py3plex.algorithms.statistics.stats_comparison import (
    compare_multilayer_networks,
    bootstrap_confidence_interval
)

## Example 1: Basic Two-Network Comparison

Let's create two simple multilayer networks and compare them.

In [ ]:
# Create Network 1: Dense social network
net1 = multinet.multi_layer_network(directed=False)
net1.add_edges([
    # Facebook layer - dense connections
    ['Alice', 'facebook', 'Bob', 'facebook', 1],
    ['Alice', 'facebook', 'Carol', 'facebook', 1],
    ['Bob', 'facebook', 'Carol', 'facebook', 1],
    ['Bob', 'facebook', 'David', 'facebook', 1],
    ['Carol', 'facebook', 'David', 'facebook', 1],
    # Twitter layer
    ['Alice', 'twitter', 'Bob', 'twitter', 1],
    ['Alice', 'twitter', 'Carol', 'twitter', 1],
    ['Bob', 'twitter', 'David', 'twitter', 1],
    # Inter-layer connections
    ['Alice', 'facebook', 'Alice', 'twitter', 1],
    ['Bob', 'facebook', 'Bob', 'twitter', 1],
], input_type='list')

print(f"Network 1: {len(list(net1.get_nodes()))} nodes, {len(list(net1.get_edges()))} edges")

In [ ]:
# Create Network 2: Sparse social network
net2 = multinet.multi_layer_network(directed=False)
net2.add_edges([
    # Facebook layer - sparse
    ['Alice', 'facebook', 'Bob', 'facebook', 1],
    ['Carol', 'facebook', 'David', 'facebook', 1],
    # Twitter layer
    ['Alice', 'twitter', 'Carol', 'twitter', 1],
    ['Bob', 'twitter', 'David', 'twitter', 1],
    # Inter-layer connections
    ['Alice', 'facebook', 'Alice', 'twitter', 1],
], input_type='list')

print(f"Network 2: {len(list(net2.get_nodes()))} nodes, {len(list(net2.get_edges()))} edges")

### Describe the Two Individual Networks

Each input is one observed graph. The comparison reports metric summaries for each graph.

In [ ]:
results = compare_multilayer_networks(
    [net1, net2], metrics=["density", "average_degree", "node_activity"]
)
columns = ["metric", "layer", "mean_group_0", "mean_group_1"]
print(results[columns].to_string(index=False))

In [ ]:
assert results["p_value"].isna().all()

## Example 2: Compare Independent Cohorts

Each cohort contains eight independently generated multiplex graphs. The Kruskal–Wallis test uses one summary per graph, metric, and layer.

In [ ]:
cohorts = [
    [
        make_random_multiplex(
            n_nodes=20, n_layers=2, p=probability,
            random_state=100 * group + replicate,
        )
        for replicate in range(8)
    ]
    for group, probability in enumerate((0.15, 0.25, 0.35))
]
results_multi = compare_multilayer_networks(
    cohorts, metrics=["density", "average_degree"], test="kruskal", correction="holm"
)
print(results_multi.to_string(index=False))

## Example 3: Directed Networks

The framework also works with directed multilayer networks.

In [ ]:
# Create directed network 1
dir_net1 = multinet.multi_layer_network(directed=True)
dir_net1.add_edges([
    ['A', 'L1', 'B', 'L1', 1],
    ['B', 'L1', 'C', 'L1', 1],
    ['C', 'L1', 'A', 'L1', 1],  # Cycle
    ['A', 'L2', 'B', 'L2', 1],
], input_type='list')

# Create directed network 2
dir_net2 = multinet.multi_layer_network(directed=True)
dir_net2.add_edges([
    ['A', 'L1', 'B', 'L1', 1],
    ['B', 'L1', 'C', 'L1', 1],  # No cycle
    ['A', 'L2', 'B', 'L2', 1],
], input_type='list')

# Compare directed networks
results_directed = compare_multilayer_networks(
    [dir_net1, dir_net2],
    metrics=['density', 'average_degree']
)

print("\nDirected Network Comparison:")
print(results_directed.to_string())

## Example 4: Bootstrap Confidence Intervals

Estimate confidence intervals for metrics using bootstrap resampling.

In [ ]:
# Define a custom metric function
def average_density(network):
    """Calculate average density across all layers."""
    from py3plex.algorithms.statistics import multilayer_statistics as mls
    
    layers = network.layers
    if not layers:
        return 0.0
    densities = [mls.layer_density(network, layer) for layer in layers]
    return np.mean(densities)

# Compute bootstrap confidence intervals
ci = bootstrap_confidence_interval(
    [net1, net2],
    average_density,
    n_bootstrap=1000,
    confidence_level=0.95, seed=42
)

print("\n95% Bootstrap Confidence Intervals for Average Density:")
for group, (lower, upper) in ci.items():
    print(f"{group}: [{lower:.4f}, {upper:.4f}]")

## Example 5: Different Statistical Tests

Demonstrate various statistical tests available.

In [ ]:
# Compare using different tests
test_types = ['permutation', 't-test', 'mann-whitney']

for test_type in test_types:
    results_test = compare_multilayer_networks(
        cohorts[:2],
        metrics=['density'],
        test=test_type,
        n_permutations=500 if test_type == 'permutation' else 1000,
        alpha=0.05
    )
    
    print(f"\n{test_type.upper()} Results:")
    print(results_test[['metric', 'layer', 'statistic', 'p_value', 'effect_size']].to_string())

## Example 6: Multiple Comparison Correction

When testing multiple metrics, apply correction for multiple comparisons.

In [ ]:
# Test many metrics
all_metrics = ['density', 'average_degree', 'clustering', 'node_activity', 'entropy']

correction_methods = ['bonferroni', 'holm', 'fdr_bh']

for correction in correction_methods:
    results_corr = compare_multilayer_networks(
        cohorts[:2],
        metrics=all_metrics,
        test='permutation',
        n_permutations=500,
        correction=correction, seed=42,
        alpha=0.05
    )
    
    n_significant = results_corr['significant'].sum()
    print(f"\n{correction.upper()}: {n_significant} significant results out of {len(results_corr)}")
    
    if n_significant > 0:
        print(results_corr[results_corr['significant']][['metric', 'p_value', 'adjusted_p_value']].to_string())

## Example 7: Check a Simulated Density Difference

Compare the range of network-level densities in the two cohorts with the lowest and highest edge probabilities.

In [ ]:
from py3plex.algorithms.statistics.multilayer_statistics import layer_density

for name, graphs in (("Lower probability", cohorts[0]), ("Higher probability", cohorts[-1])):
    densities = [layer_density(graph, graph.layers[0]) for graph in graphs]
    print(f"{name}: mean {np.mean(densities):.3f}, range {min(densities):.3f}–{max(densities):.3f}")

## Summary

- Pass a flat list of networks to describe the observed graphs.
- Pass independent graph cohorts to perform inferential comparisons.
- Choose tests based on the study design and their assumptions. A permutation test requires exchangeable group assignments under the null.
- Adjust p-values when testing several metrics or layers and report effect sizes.
- An edge bootstrap estimates variability within each graph under edge resampling.